# Verify a design against Cameo requirements

Verification driven by **requirements extracted from a Cameo (`.mdzip`) model in Istari**. Nothing in the pass/fail logic is hard-coded: the limits come out of the SysML requirement text, so the authoritative source of truth stays in the systems model.

1. Read `requirements.json` from a Cameo extraction already stored in Istari
2. Check a design parameter set against those requirement limits
3. Upload the outputs and log a `FAILED` entry
4. Fix the two violations, re-run, and log `SUCCESS`
5. Attach the passing report to the design model as an artifact

Companion to [Scenario A quickstart](workflow_log_scenario_a_simple.ipynb), which uses the same workflow-log mechanics with a synthetic stress check instead of real requirements.

### Prerequisites

Run these in a terminal from the cookbook root **before** starting this notebook — the kernel has to exist before the notebook can attach to it:

```bash
uv sync --group dev
```

```bash
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Then pick the **Python (istari-client-cookbook)** kernel. Only the `dev` group is needed.

Also required:

- **Registry Service > 10.17.3** (2026-05 release or later)
- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
- **Experimental features** enabled in the web app (Istari Digital → **Application Settings**) so the **Workflow log** tab is visible
- A Cameo `.mdzip` model on your instance that has already been extracted — see below

### Where `requirements.json` comes from

The Cameo integration writes it as a job product. If your instance has no extraction yet, produce one with a Cameo agent:

```python
job = client.add_job(model_id=MDZIP_MODEL_ID, function="@istari:extract", tool_name="dassault_cameo")
```

Poll until `COMPLETED`, and `requirements.json` appears among the model's artifacts. [Cameo requirements extraction + tag update](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb) walks through that end to end, including writing values back into the model with `@istari:update_tags`. This notebook only *consumes* the extraction, so it needs no Cameo agent of its own.

## 1. Connect

`Client` reads models, artifacts, and files; `V3Client` handles workflow outputs and workflow log entries.

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv

from istari_helpers import commit_changes

from istari_digital_client import Client, Configuration, V3Client
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]

config = Configuration(
    registry_url=REGISTRY_URL,
    registry_auth_token=os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"],
)
client = Client(config)   # models, artifacts, files, systems
v3 = V3Client(config)     # workflow outputs + workflow log

UI_URL = REGISTRY_URL.rstrip("/").replace("//fileservice-v2.", "//")

print("Registry:", client.check_compatibility().server_version)
print("Web app:", UI_URL)

## 2. Read the requirements out of Istari

Find a Cameo model that has an extracted `requirements.json` artifact and read its latest revision. `parse_limits` pulls the numeric bounds out of the SysML requirement text — the two phrasings this model uses are *"shall be between X mm and Y mm"* and *"shall not exceed X mm"*.

The search accepts the first extraction that contains at least three numerically bounded requirements, which skips stale extractions from before the requirements were written.

In [ ]:
MDZIP_NAME = "NCXTable-example.mdzip"  # Cameo model to verify against
MAX_MODELS_SCANNED = 200               # bound the search on a busy instance


def parse_limits(text):
    """Return (low, high) mm bounds from SysML requirement text, or None if not numeric.

    Either bound may be None, meaning unbounded on that side.
    """
    between = re.search(r"between\s+([\d.]+)\s*mm\s+and\s+([\d.]+)\s*mm", text, re.IGNORECASE)
    if between:
        return float(between.group(1)), float(between.group(2))
    at_most = re.search(r"shall not exceed\s+([\d.]+)\s*mm", text, re.IGNORECASE)
    if at_most:
        return None, float(at_most.group(1))
    return None


def find_extracted_requirements(mdzip_name, min_bounded=3):
    """First (model, artifact, requirements) whose extraction has numeric requirements."""
    for scanned, model in enumerate(client.list_models(size=100).iter_items()):
        if scanned >= MAX_MODELS_SCANNED:
            break
        if not model.file or model.file.name != mdzip_name:
            continue
        for artifact in client.list_model_artifacts(model.id, size=50).items:
            if not artifact.file or artifact.file.name != "requirements.json":
                continue
            raw = artifact.file.revisions[-1].read_bytes()
            requirements = json.loads(raw.decode("utf-8"))
            bounded = sum(1 for r in requirements if parse_limits(r.get("text") or ""))
            if bounded >= min_bounded:
                return model, artifact, requirements, raw
    return None, None, None, None


mdzip_model, req_artifact, requirements, REQUIREMENTS_BYTES = find_extracted_requirements(MDZIP_NAME)
if req_artifact is None:
    raise RuntimeError(
        f"No extracted requirements.json found on a {MDZIP_NAME!r} model. Run "
        '@istari:extract with tool_name="dassault_cameo" on the Cameo model first '
        "(see the prerequisites above), or set MDZIP_NAME to a model you have."
    )

req_revision = req_artifact.file.revisions[-1]
print(f"Cameo model:  {mdzip_model.file.name}  (model {mdzip_model.id})")
print(f"Requirements: artifact {req_artifact.id}, revision {req_revision.id}")
print(f"{len(requirements)} requirements extracted; numerically bounded ones:\n")

for req in requirements:
    limits = parse_limits(req.get("text") or "")
    if limits:
        low, high = limits
        window = f"{low:g}-{high:g} mm" if low is not None else f"<= {high:g} mm"
        print(f"  [{req['req_id']}] {req['name']:<38} {window}")

## 3. Create the system for the design under test

The design is a plain JSON parameter set — the numbers a CAD build or an analysis would produce — tracked as one file on one configuration. Each run creates a fresh system; the teardown cell at the bottom archives it.

`commit_changes` (from [`istari_helpers.py`](istari_helpers.py)) snapshots the configuration and advances the baseline tag. A workflow log entry can only reference a configuration that is in the branch history, so this is required here and again after every design revision.

In [ ]:
from istari_digital_client.v2.models.new_system import NewSystem
from istari_digital_client.v2.models.new_system_configuration import NewSystemConfiguration
from istari_digital_client.v2.models.new_tracked_file import NewTrackedFile
from istari_digital_client.v2.models.tracked_file_specifier_type import TrackedFileSpecifierType

SYSTEM_NAME = "Cameo Requirements Verification"
CONFIG_NAME = "requirements-check"

# PLATE_1 thickness is under the 26 mm floor and CUBE_1 overhangs the plate: two violations.
INITIAL_DESIGN = {
    "PLATE_1": {"thickness_mm": 24.0, "length_mm": 140.0, "width_mm": 140.0},
    "CUBE_1": {"height_mm": 100.0, "length_mm": 120.0, "width_mm": 105.0},
}

work = Path("_cameo_req_run")
work.mkdir(exist_ok=True)
design = work / "plate-cube-design.json"
design.write_text(json.dumps(INITIAL_DESIGN, indent=2))

model = client.add_model(
    path=design,
    display_name=design.name,
    description="Plate/cube assembly - initial design",
)
system = client.create_system(
    NewSystem(
        name=SYSTEM_NAME,
        description=f"Design verified against requirements from {mdzip_model.file.name}",
    )
)
configuration = client.create_configuration(
    system_id=system.id,
    new_system_configuration=NewSystemConfiguration(
        name=CONFIG_NAME,
        tracked_files=[
            NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LATEST,
                file_id=model.file.id,
            )
        ],
    ),
)

SYSTEM_ID = system.id
CONFIG_ID = configuration.id
MODEL_ID = model.id
FILE_ID = model.file.id

commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("Created system:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 4. Verify the design against the requirements

`PARAM_FOR_REQ` is the verification matrix: which design parameter satisfies which Cameo requirement. Everything else — the limits, the requirement IDs on each result — comes from the extraction, so editing a bound in Cameo and re-extracting changes the verdict here without touching this notebook.

Each run writes three output files, including the exact requirement set it was checked against.

In [ ]:
# Which design parameter each Cameo requirement constrains.
PARAM_FOR_REQ = {
    "Requirement REQ-001 Plate Thickness": ("PLATE_1", "thickness_mm"),
    "Requirement REQ-002 Plate Length": ("PLATE_1", "length_mm"),
    "Requirement REQ-003 Plate Width": ("PLATE_1", "width_mm"),
    "Requirement REQ-004 Cube Height": ("CUBE_1", "height_mm"),
    "Requirement REQ-005 Cube Length": ("CUBE_1", "length_mm"),
    "Requirement REQ-006 Cube Width": ("CUBE_1", "width_mm"),
}


def run_checks(design_bytes, out_dir):
    """Verify every mapped requirement, write the outputs, return (verdict, paths)."""
    values = json.loads(design_bytes)
    checks = []

    for req in requirements:
        target = PARAM_FOR_REQ.get(req["name"])
        limits = parse_limits(req.get("text") or "")
        if target is None or limits is None:
            continue  # narrative or unmapped requirement - not machine-verifiable here

        part, parameter = target
        value = values[part][parameter]
        low, high = limits
        passed = (low is None or value >= low) and (high is None or value <= high)
        window = f"{low:g}-{high:g} mm" if low is not None else f"<= {high:g} mm"

        checks.append({
            "req_id": req["req_id"],
            "requirement": req["name"],
            "parameter": f"{part}.{parameter}",
            "value_mm": value,
            "limits_mm": [low, high],
            "passed": passed,
            "detail": f"{value:g} mm against {window}",
        })

    if not checks:
        raise RuntimeError(
            "No requirement matched PARAM_FOR_REQ. Extracted names: "
            f"{[r['name'] for r in requirements]}"
        )

    verdict = "SUCCESS" if all(check["passed"] for check in checks) else "FAILED"

    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / "results.json"
    report_path = out_dir / "report.txt"
    requirements_path = out_dir / "requirements.json"

    results_path.write_text(json.dumps({
        "verdict": verdict,
        "requirements_source": {
            "cameo_model": mdzip_model.file.name,
            "model_id": mdzip_model.id,
            "artifact_id": req_artifact.id,
            "revision_id": req_revision.id,
        },
        "checks": checks,
    }, indent=2))
    report_path.write_text(
        "".join(
            f"{'PASS' if check['passed'] else 'FAIL'}  [{check['req_id']}] "
            f"{check['parameter']:<24}{check['detail']}\n"
            for check in checks
        )
    )
    requirements_path.write_bytes(REQUIREMENTS_BYTES)  # what we verified against

    print(report_path.read_text(), end="")
    print("Verdict:", verdict)
    return verdict, [results_path, report_path, requirements_path]


source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
verdict, output_paths = run_checks(source_bytes, work / "iter-1")

## 5. Upload the outputs and record the log entry

`create_workflow_output` registers each result file against the system; `create_workflow_log_entry` ties the title, `status`, `configuration_id`, and output IDs into one durable record. Both iterations use the same helper.

> **Pause here.** Open the system in the web app → **Workflow log** tab, open the failed entry, and preview the attached outputs — including the requirement set the run was judged against.

In [ ]:
def log_run(title, verdict, paths):
    """Upload each output file, then record one workflow log entry."""
    output_ids = [v3.create_workflow_output(system_id=SYSTEM_ID, path=p).id for p in paths]
    entry = v3.create_workflow_log_entry(
        system_id=SYSTEM_ID,
        workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
            title=title,
            status=verdict,  # SUCCESS | FAILED | UNSPECIFIED
            configuration_id=CONFIG_ID,
            workflow_output_ids=output_ids,
        ),
    )
    print(f"{entry.status}  {title}  ({len(output_ids)} outputs, entry {entry.id})")
    return entry


entry1 = log_run("Requirements verification - iter-1 (initial design)", verdict, output_paths)
print("Workflow log tab:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 6. Fix the violations, re-run, and log the pass

Thicken the plate to 28 mm to clear REQ-001 and shorten the cube to 105 mm so it no longer overhangs, per REQ-005. The verification code is unchanged — `get_file` returns the latest revision, so the second run picks up the new numbers.

In [ ]:
REVISED_DESIGN = {
    "PLATE_1": {"thickness_mm": 28.0, "length_mm": 140.0, "width_mm": 140.0},
    "CUBE_1": {"height_mm": 100.0, "length_mm": 105.0, "width_mm": 105.0},
}
design.write_text(json.dumps(REVISED_DESIGN, indent=2))
client.update_model(
    model_id=MODEL_ID,
    path=design,
    description="Plate/cube assembly - REQ-001 and REQ-005 fixed",
    version_name="v2-req-fixes",
)
commit_changes(client, SYSTEM_ID, CONFIG_ID)

source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
verdict, output_paths = run_checks(source_bytes, work / "iter-2")

entry2 = log_run("Requirements verification - iter-2 (REQ-001, REQ-005 fixed)", verdict, output_paths)
print("Review both entries:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 7. Attach the verified results to the design model

`create_resource(resource_type=artifact)` uploads the passing report — the V3 Resources equivalent of `Client.add_artifact()` — and a `produces` revision relationship links the design model revision to it. Workflow outputs stay outside configuration snapshots; a model-linked artifact does not, so one more `commit_changes` puts it on the baseline snapshot.

In [ ]:
import shutil

from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto
from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

# The passing report from iter-2, renamed to say what it is.
verified = work / f"verified-results-for-{design.name}"
shutil.copyfile(work / "iter-2" / "results.json", verified)

artifact = v3.create_resource(
    path=verified,
    resource_type=ResourceTypeDto.ARTIFACT,
    display_name=verified.name,
    description=f"Verified against {mdzip_model.file.name} requirements (entry {entry2.id})",
)

model_resource = v3.get_resource(resource_id=MODEL_ID)
produces = next(t for t in v3.list_revision_relationship_types().items if t.name == "produces")
v3.create_revision_relationship(
    new_revision_relationship_dto=NewRevisionRelationshipDto(
        relationship_type_id=produces.id,
        left_revision_id=model_resource.file_revision_id,
        right_revision_id=artifact.file_revision_id,
    ),
)
commit_changes(client, SYSTEM_ID, CONFIG_ID)

print(f"Artifact {artifact.resource_id} linked to model {MODEL_ID} via {produces.name}")
print("Committed to the baseline snapshot")

## Recap

On one system you now have:

- Two workflow log entries — `FAILED`, then `SUCCESS` — each linked to the `requirements-check` configuration
- Every result file from both runs, including the exact `requirements.json` revision each verdict was judged against
- The verified report as an artifact resource on the design model via a `produces` relationship, committed to the baseline snapshot

The limits were never written down here. They live in the Cameo model, reach this notebook through `@istari:extract`, and each result carries the `req_id` it satisfies — so a requirement change in Cameo, re-extracted, flips the verdict on the next run with no code change.

### Learn more

- [External workflow logs](https://docs.istaridigital.com/developers/SDK/v3/03-workflow-logs) - `create_workflow_output`, `create_workflow_log_entry`, and listing entries
- [Cameo requirements extraction + tag update](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb) - producing `requirements.json` and writing values back with `@istari:update_tags`
- [Scenario A quickstart](workflow_log_scenario_a_simple.ipynb) - the same workflow-log flow with a synthetic check
- [Scenario B](workflow_log_scenario_b.ipynb) - logging a tradespace sweep

## Teardown

Archive the demo system so repeated runs do not clutter the instance. The Cameo model and its extraction are left untouched — this notebook only read them. Archiving is reversible.

In [ ]:
client.archive_system(system_id=SYSTEM_ID)
print("Archived system", SYSTEM_ID)